<a href="https://colab.research.google.com/github/kolaparthisaimeenakshi-crypto/ai-mentor-portfolia/blob/main/Day6_placement_processor.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [34]:
!pip install -q google-genai pydantic
import os, getpass

os.environ['GEMINI_API_KEY'] = getpass.getpass(
    'Enter NEW Gemini API key: '
)

Enter NEW Gemini API key: ··········


In [35]:
from pydantic import BaseModel
from typing import List, Optional

class Education(BaseModel):
    degree: str
    institution: str
    year: int

class Resume(BaseModel):
    name: str
    email: str
    phone: Optional[str] = None
    education: List[Education]
    skills: List[str]
    projects: List[str] = []
    experience_years: float

In [36]:
from google import genai
from pydantic import ValidationError

client = genai.Client(api_key=os.environ['GEMINI_API_KEY'])

def extract_resume(raw_text: str, max_retries: int = 1) -> Resume:
    """Extract a Resume JSON from raw text. Retries once on schema fail."""
    for attempt in range(max_retries + 1):
        try:
            resp = client.models.generate_content(
                model='gemini-2.5-flash',
                contents=f'Extract a Resume JSON from this text. Return ONLY JSON, no markdown.\n\n{raw_text}',
                config={
                    'response_mime_type': 'application/json',
                    'response_schema': Resume.model_json_schema(),
                },
            )
            return Resume.model_validate_json(resp.text)
        except ValidationError as e:
            if attempt == max_retries:
                raise
            fix_prompt = f'Fix this JSON to match schema. Errors: {e}. Original: {resp.text}'
            resp = client.models.generate_content(
                model='gemini-2.5-flash', contents=fix_prompt,
                config={'response_mime_type': 'application/json',
                        'response_schema': Resume.model_json_schema()})
            return Resume.model_validate_json(resp.text)

In [37]:
with open('../content/sample_data/sample_resumes.txt') as f:
    resumes = [r.strip() for r in f.read().split('---') if r.strip()]
print(f'Loaded {len(resumes)} sample résumés')

results = []
errors = []
for i, r in enumerate(resumes):
    try:
        parsed = extract_resume(r)
        results.append(parsed)
        print(f'  [{i+1}] {parsed.name} — {len(parsed.skills)} skills')
    except Exception as e:
        errors.append((i, e))
        print(f'  [{i+1}] FAILED: {type(e).__name__}: {str(e)[:120]}')

print(f'\n{len(results)}/5 succeeded, {len(errors)} failed')

Loaded 5 sample résumés
  [1] Ravi Kumar — 6 skills
  [2] Sneha Reddy — 6 skills
  [3] Arun Pillai — 8 skills
  [4] Priya Nair — 5 skills
  [5] Karthik Sharma — 5 skills

5/5 succeeded, 0 failed


In [38]:
# Empty string
try:
    bad = extract_resume('')
    print('Unexpected success:', bad.model_dump_json())
except Exception as e:
    print(f'Empty input: {type(e).__name__}: {str(e)[:200]}')

# Whitespace only
try:
    bad = extract_resume('   \n\n   ')
    print('Unexpected success:', bad.model_dump_json())
except Exception as e:
    print(f'Whitespace input: {type(e).__name__}: {str(e)[:200]}')

# Garbage non-résumé text
try:
    bad = extract_resume('the quick brown fox jumps over the lazy dog')
    print('Garbage input:', bad.model_dump_json())
except Exception as e:
    print(f'Garbage input: {type(e).__name__}: {str(e)[:200]}')

Unexpected success: {"name":"John Doe","email":"john.doe@example.com","phone":null,"education":[{"degree":"B.S. Computer Science","institution":"University of Placeholder","year":2020}],"skills":["Python","JavaScript","SQL"],"projects":[],"experience_years":3.5}
Unexpected success: {"name":"","email":"","phone":null,"education":[],"skills":[],"projects":[],"experience_years":0.0}
Garbage input: ClientError: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.


In [39]:
class JD(BaseModel):
    company: str
    role: str
    must_have_skills: List[str]
    nice_to_have_skills: List[str] = []
    min_cgpa: Optional[float] = None
    locations: List[str] = []
    package_lpa: Optional[float] = None

In [40]:
import requests
from bs4 import BeautifulSoup
import pathlib, json

def fetch_jd(url, max_chars=6000):
    """Fetch JD URL and return clean text. Returns None on block / failure."""
    try:
        r = requests.get(url, headers={'User-Agent': 'Mozilla/5.0'}, timeout=10)
        r.raise_for_status()
        soup = BeautifulSoup(r.text, 'html.parser')
        # Remove script and style tags
        for tag in soup(['script', 'style']):
            tag.decompose()
        return soup.get_text(separator='\n', strip=True)[:max_chars]
    except Exception as e:
        print(f'  Scrape failed for {url}: {e}')
        return None

# Test on one URL
test_url = 'https://amazon.jobs/en/jobs/10386843/software-dev-engineer-ii'
text = fetch_jd(test_url)
if text:
    print(f'Got {len(text)} chars')
    print(text[:300])
else:
    print('Scrape blocked. Will use cached set.')

Got 5213 chars
Software Dev Engineer II - Job ID: 10386843 | Amazon.jobs
Skip to main content
×
Home
Teams
Locations
Job categories
My career
My applications
My profile
Account security
Settings
Sign out
Resources
Accommodations
Benefits
Inclusive experiences
How We Hire
Leadership principles
Working at Amazon
FAQ


In [41]:
def normalise_jd(text: str) -> JD:
    """Send JD text to Gemini, get structured JD JSON back."""
    resp = client.models.generate_content(
        model='gemini-2.5-flash',
        contents=f'Extract a JD JSON from this text:\n\n{text}',
        config={
            'response_mime_type': 'application/json',
            'response_schema': JD.model_json_schema(),
        },
    )
    return JD.model_validate_json(resp.text)

# Test on one JD text
if text:
    jd = normalise_jd(text)
    print(jd.model_dump_json(indent=2))

{
  "company": "Amazon",
  "role": "Software Dev Engineer II",
  "must_have_skills": [
    "Professional software development experience (3+ years)",
    "System design and architecture experience (2+ years)",
    "Programming with at least one software programming language",
    "Algorithms",
    "Object-oriented design",
    "Distributed systems",
    "Web development",
    "Design patterns",
    "Efficient coding practices",
    "Building Web Services",
    "Building APIs",
    "Building Micro-Services",
    "Troubleshooting skills",
    "Strong ownership",
    "Customer experience focus"
  ],
  "nice_to_have_skills": [
    "Full software development life cycle experience (3+ years)",
    "Coding standards",
    "Code reviews",
    "Source control management",
    "Build processes",
    "Testing",
    "Operations experience",
    "Bachelor's degree in computer science or equivalent",
    "Experience with building web-based applications"
  ],
  "min_cgpa": null,
  "locations": [
    

### Fix for `NameError: name 'fetch_jd' is not defined`

If you encounter a `NameError` when running the cell below, it means the functions `fetch_jd` and `normalise_jd` have not been defined in the current kernel session.

**To fix this, please run the following cells in the notebook before executing the next cell:**

1.  **Cell `-aNZiP06lkj7`**: This cell defines the `fetch_jd` function.
2.  **Cell `3YRM2EAvmYZW`**: This cell defines the `normalise_jd` function.

Ensure these cells execute successfully before attempting to run the data processing loop.

In [42]:
import json, pathlib

URLS = [
    # Paste your 5 assigned URLs here
    'https://amazon.jobs/en/jobs/10386843/software-dev-engineer-ii',
    'https://amazon.jobs/en/jobs/10429151/salesforce-developer-dsp-tech-last-mile',
    'https://www.capgemini.com/in-en/jobs/453071-en_GB+sap_btp',
    'https://www.capgemini.com/in-en/jobs/473200-en_GB+sap_btp',
    'https://careers.wipro.com/job/AUTOMATION-DEVELOPER-%28BUSINESS-PROCESS-&amp;-ITSM-AUTOMATION%29/154629-en_US',
]

CACHE = pathlib.Path('../data/jds_cached.jsonl')
USE_CACHE = False   # set True if scraping is blocked

jds = []

if USE_CACHE and CACHE.exists():
    print(f'Using cached JDs from {CACHE}')
    for line in CACHE.read_text().splitlines():
        jds.append(JD.model_validate_json(line))
else:
    for url in URLS:
        text = fetch_jd(url)
        if text is None:
            continue
        try:
            jd = normalise_jd(text)
            jds.append(jd)
            print(f'  ✓ {jd.company} — {jd.role}')
        except Exception as e:
            print(f'  ✗ {url}: {e}')

print(f'\nProcessed {len(jds)} JDs')

# Inspect first 3
for jd in jds[:3]:
    print(f'\n{jd.company} - {jd.role}')
    print(f'  Must: {jd.must_have_skills}')
    print(f'  Nice: {jd.nice_to_have_skills}')
    print(f'  CGPA: {jd.min_cgpa}, LPA: {jd.package_lpa}')

  ✓ Amazon — Software Dev Engineer II
  ✓ Amazon — Salesforce Developer
  ✓ Capgemini — Application Developer
  ✓ Capgemini — Cloud Transformation Architect
  ✓ Wipro Limited — AUTOMATION DEVELOPER (BUSINESS PROCESS & ITSM AUTOMATION)

Processed 5 JDs

Amazon - Software Dev Engineer II
  Must: ['3+ years of non-internship professional software development experience', '2+ years of non-internship design or architecture experience (design patterns, reliability and scaling)', 'Experience programming with at least one software programming language', 'Sound understanding of computer science', 'Algorithms', 'Object-oriented design', 'Distributed systems', 'Web development', 'Front-end/back-end design', 'Strong ownership', 'Excellent troubleshooting skills']
  Nice: ['3+ years of full software development life cycle, including coding standards, code reviews, source control management, build processes, testing, and operations experience', 'Experience with building web-based applications and/or

In [43]:
OUT = pathlib.Path('data/jds.jsonl')
OUT.parent.mkdir(exist_ok=True)
with open(OUT, 'w') as f:
    for jd in jds:
        f.write(jd.model_dump_json() + '\n')
print(f'Wrote {len(jds)} JDs to {OUT}')

# Verify the file
with open(OUT) as f:
    for line in f:
        d = json.loads(line)
        print(f'  {d["company"]:20} | {d["role"]:30} | {len(d["must_have_skills"])} must-haves')

Wrote 5 JDs to data/jds.jsonl
  Amazon               | Software Dev Engineer II       | 11 must-haves
  Amazon               | Salesforce Developer           | 11 must-haves
  Capgemini            | Application Developer          | 0 must-haves
  Capgemini            | Cloud Transformation Architect | 0 must-haves
  Wipro Limited        | AUTOMATION DEVELOPER (BUSINESS PROCESS & ITSM AUTOMATION) | 1 must-haves
